In [1]:
# Explicitly Point Spark to the Correct Python Executable
# If you already have numpy installed in a specific environment (like conda or a virtualenv), but PySpark is defaulting to /usr/bin/python3 or system Python, 
# explicitly set the Python environment variables before initializing SparkContext:

import os
import sys

# Set these before initializing SparkContext (sc)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession. \
    builder. \
    enableHiveSupport(). \
    appName('Window-Function-Demo'). \
    master('yarn'). \
    getOrCreate()

In [4]:
# spark = SparkSession. \
#     builder. \
#     enableHiveSupport(). \
#     appName('Date-Time-Demo'). \
#     master('local[*]'). \
#     getOrCreate()

In [5]:
sc = spark.sparkContext

# pyspark.sql.functions.to_json

<img src="images/to_json.jpg" align = "left">



Ref -https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.to_json.html

Where `from_json` is used to unpack data, `to_json` does the exact opposite: it takes a structured column (like a `StructType`, an `ArrayType`, or a `MapType`) and packs it into a single JSON string.

This is incredibly useful when you need to send data to a downstream system that expects a specific JSON format, such as pushing records to a **Kafka topic** or saving data to a **NoSQL** database like MongoDB.

# 1. How it Works

The function converts a complex object into a standard JSON string.

1. **Input:** A column of type Struct, Array, or Map.

2. **Logic:** Spark serializes the key-value pairs into a string format.

3. **Output:** A single string column.

# 2. Working Example

Imagine you have a clean table with multiple columns, and you want to bundle the user details into a single JSON string for an API export.

**Step 1:** Create Structured Data  
We'll use struct to group individual columns together first.

In [4]:
from pyspark.sql.functions import to_json, struct, col

data = [
    (1, "Alice", 30, "New York"),
    (2, "Bob", 25, "Seattle")
]
df = spark.createDataFrame(data, ["id", "name", "age", "city"])

# Group columns into a single Struct named 'user_info'
df_structured = df.select("id", struct("name", "age", "city").alias("user_info"))
# print(df_structured.printSchema())

root
 |-- id: long (nullable = true)
 |-- user_info: struct (nullable = false)
 |    |-- name: string (nullable = true)
 |    |-- age: long (nullable = true)
 |    |-- city: string (nullable = true)

None


**Step 2:** Apply `to_json`

Now, we convert that user_info struct into a JSON string.

In [5]:
# Convert the Struct column to a JSON String
json_df = df_structured.withColumn("json_string", to_json(col("user_info")))

json_df.select("id", "json_string").show(truncate=False)

+---+-------------------------------------------+
|id |json_string                                |
+---+-------------------------------------------+
|1  |{"name":"Alice","age":30,"city":"New York"}|
|2  |{"name":"Bob","age":25,"city":"Seattle"}   |
+---+-------------------------------------------+



# 3. Key Details to Remember

- **Structs are best:** While you can use to_json on Maps, using struct() allows you to mix different data types (strings, ints, booleans) into the same JSON object.

- **Options:** You can pass a map of options to customize the output, such as {"timestampFormat": "yyyy-MM-dd"} to control how dates look inside the string.

- **Kafka Integration:** If you are writing to Kafka, your DataFrame must have a column named value of type string or binary. `to_json` is the standard way to prepare your data for that value column.

<img src="images/summary_comparison1.jpg" align = "left">